<a href="https://colab.research.google.com/github/PlaZMaD/MLDM-2026/blob/main/02-linear-regression/LinearRegression_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42  # fix everything to this seed, so that your results are reproducible

## Task 1 (3 points)

Consider the following toy dataset. Note that the data generator takes an explicit
`seed`: re-running the cell always gives you **the same** points and the same outliers,
so the numbers you report in Task 1 and Task 2 are directly comparable.

In [ ]:
true_function = lambda x: 3.25 * x - 1.47
W_TRUE = np.array([3.25, -1.47])  # (slope, intercept), used for sanity checks below

def noise_function(x, rng):
    return (
        rng.normal(size=len(x)) * 4. +
        np.where(
            x < 15,
            0.,
            -60 + rng.normal(size=len(x)) * 10
        ) * (rng.uniform(size=len(x)) < 0.4).astype(int)
    )

limits = (-4., 20.)

def generate_dataset(N=300, seed=SEED):
    rng = np.random.default_rng(seed)
    x = rng.uniform(*limits, size=N)
    y = true_function(x) + noise_function(x, rng)

    return x[:, None], y

X, y = generate_dataset()

# Hold out a test set: all the metrics below are reported on data the models never saw.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED
)

x = np.linspace(*limits, 101)
plt.plot(x, true_function(x), c='black', label='True dependence')
plt.scatter(X_train, y_train, s=10, label='Train (noise + outliers)')
plt.scatter(X_test, y_test, s=10, label='Test')
plt.legend();

Implement a linear model $\hat y(x|w) = w_1\cdot x + w_0$ with MAE loss:
$$\text{MAE}(w) = \frac{1}{N}\sum_{i=1}^N\left|y_i - \hat y(x_i|w)\right|,$$
using gradient descent optimization in `numpy`. Fit on the **train** part only.

*Hint: introduce a constant feature to account for the bias term to make the formulas simpler.*

### Hints on the optimization

* MAE is not differentiable at zero, so what you compute is a **subgradient**. With
  $\hat y = Xw$ and residuals $r_i = y_i - \hat y_i$:
  $$\frac{\partial \text{MAE}}{\partial w} = -\frac{1}{N} X^\top \text{sign}(r).$$
  `np.sign(0) == 0`, which is a perfectly valid subgradient choice.
* Unlike MSE, the MAE gradient does not shrink as you approach the optimum: its magnitude
  stays $O(1)$. A constant learning rate therefore makes the weights oscillate around the
  minimum forever. Either decay the learning rate (e.g. `lr_t = lr / (1 + t / t0)`) or stop
  on a tolerance, and say in your comment which one you used and why.
* $x \in [-4, 20]$ while the bias feature is constantly $1$, so the two components of the
  gradient live on very different scales and plain GD converges slowly. Standardizing $x$
  before the fit (and converting the weights back afterwards) helps a lot. Try it both ways.

### What to report

1. The learning curve (loss vs. iteration) for **at least three learning rates** on one plot,
   with a comment on what too large / too small looks like.
2. The final fit together with `sklearn`'s `LinearRegression` (fitted on the same train set)
   and the true dependence, on one plot.
3. The metrics table from the `compare_models` helper below. Then answer:
   * which model is closer to the true coefficients, and why?
   * the MAE-fitted model may well lose to `LinearRegression` in test **MSE**. Is that a bug?
     Explain what each loss is actually optimizing in the presence of the outliers at $x > 15$.

In [ ]:
def check_gradient(loss_fn, grad_fn, w, eps=1e-5, tol=1e-4):
    """Compare an analytical gradient against a central finite difference.

    Call this on your own loss/gradient before debugging the training loop --
    most 'my GD does not converge' bugs are actually gradient bugs.
    """
    w = np.asarray(w, dtype=float)
    analytic = np.asarray(grad_fn(w), dtype=float)
    numeric = np.empty_like(w)
    for k in range(w.size):
        step = np.zeros_like(w)
        step[k] = eps
        numeric[k] = (loss_fn(w + step) - loss_fn(w - step)) / (2 * eps)

    max_diff = np.abs(analytic - numeric).max()
    assert max_diff < tol, (
        f"gradient mismatch: analytic={analytic}, numeric={numeric}, max diff={max_diff:.2e}"
    )
    print(f"gradient OK (max diff = {max_diff:.2e})")
    return max_diff


def compare_models(models, X_test, y_test, w_true=W_TRUE):
    """models: dict {name: fitted model with .predict(X) and .coef_ / .intercept_}.

    Returns a table with test MAE / MSE and the distance to the true coefficients.
    """
    rows = []
    for name, model in models.items():
        pred = model.predict(X_test)
        w = np.array([np.ravel(model.coef_)[0], np.ravel(model.intercept_)[0]])
        rows.append({
            'model': name,
            'test MAE': mean_absolute_error(y_test, pred),
            'test MSE': mean_squared_error(y_test, pred),
            'w1 (true 3.25)': w[0],
            'w0 (true -1.47)': w[1],
            '||w - w_true||': np.linalg.norm(w - np.asarray(w_true)),
        })
    return pd.DataFrame(rows).set_index('model').round(3)

In [ ]:
<YOUR CODE>

Now compare the two fits quantitatively (numbers, not eyeballing the plot).
Wrap your gradient-descent solution into anything with a `predict` method and
`coef_` / `intercept_` attributes, or just build the table by hand in the same format:

```python
compare_models({'GD + MAE': my_model, 'sklearn (OLS)': sk_model}, X_test, y_test)
```

In [ ]:
<YOUR CODE>

## Task 2 (2 + 1 points)

* Wrap your solution from the previous task into a class. Plot the learning curve and the
  final fit. Compare and comment your results with the previous ones -- since the data is
  seeded, the two fits should agree up to the optimization tolerance; if they do not, say why.
  **(2 points)**

* Make it possible to choose a loss function. **(1 point)**

Keep the class `sklearn`-compatible:

* `__init__` must only store its arguments as attributes **with the same names**, and must not
  do any validation or computation. That is what `BaseEstimator.get_params` relies on, and
  without it `clone`, `cross_val_score` and `GridSearchCV` will not work with your estimator.
* Everything learned in `fit` goes to attributes with a trailing underscore (`coef_`,
  `intercept_`, `loss_history_`), and `fit` returns `self`.
* `score` comes for free from `RegressorMixin` -- no need to implement it. `fit_predict` is a
  transformer/clusterer convention, not a regressor one, so it is dropped here.

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import cross_val_score
from sklearn.utils.estimator_checks import check_estimator

In [ ]:
class LinearRegressionGD(BaseEstimator, RegressorMixin):
    """Linear regression fitted by (full-batch) gradient descent.

    Parameters
    ----------
    loss : {'mae', 'mse'}, default='mae'
        Loss to minimize. Feel free to add more (e.g. 'huber').
    lr : float, default=0.01
        Learning rate.
    n_iter : int, default=1000
        Maximum number of gradient steps.
    tol : float, default=1e-6
        Stop when the loss improvement falls below this value.
    fit_intercept : bool, default=True
        Whether to add the constant feature.
    """

    def __init__(self, loss='mae', lr=0.01, n_iter=1000, tol=1e-6, fit_intercept=True):
        # only assignments here -- no validation, no computation
        self.loss = loss
        self.lr = lr
        self.n_iter = n_iter
        self.tol = tol
        self.fit_intercept = fit_intercept

    def fit(self, X, y):
        # set self.coef_, self.intercept_, self.loss_history_ and return self
        raise NotImplementedError

    def predict(self, X):
        raise NotImplementedError

In [ ]:
<YOUR CODE>

Self-check: if the class is implemented correctly, both cells below run without errors.
`cross_val_score` internally clones your estimator, so it fails loudly whenever `__init__`
does not follow the `sklearn` convention.

In [ ]:
model = LinearRegressionGD(loss='mae').fit(X_train, y_train)
print('CV R^2:', cross_val_score(LinearRegressionGD(loss='mae'), X, y, cv=5))

compare_models(
    {
        'GD + MAE (class)': model,
        'GD + MSE (class)': LinearRegressionGD(loss='mse').fit(X_train, y_train),
        'sklearn (OLS)': LinearRegression().fit(X_train, y_train),
    },
    X_test, y_test,
)

In [ ]:
# Optional, but a good habit: the full sklearn estimator contract.
# check_estimator(LinearRegressionGD())